In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_used_cars.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (61056, 15)


,name,year,fuel,transmission,registration_location,color,assembly,body_type,price_pkr,mileage_km,engine_cc,battery_kwh,feature_count,brand,model
0,Suzuki Alto VXL AGS 2022,2022,Petrol,Automatic,Islamabad,Solid White,Local,Hatchback,2390000.0,120000.0,660.0,NaN,10,Suzuki,Alto
1,Honda N Box Custom GL 2022,2022,Petrol,Automatic,Islamabad,White,Imported,Hatchback,3340000.0,37110.0,658.0,NaN,20,Honda,N
2,Honda Civic EX 1995,1995,LPG,Manual,Karachi,Black,Local,Sedan,630000.0,786.0,1500.0,NaN,8,Honda,Civic
3,Toyota Corolla GLi Automatic 1.6 VVTi 2012,2012,Petrol,Automatic,Lahore,Medium Silver,Imported,Sedan,3325000.0,133000.0,1600.0,NaN,9,Toyota,Corolla
4,Toyota Corolla Hatchback 1998,1998,Petrol,Automatic,Punjab,Black,Local,Unknown,1645000.0,125225.0,1600.0,NaN,15,Toyota,Corolla


In [2]:
REFERENCE_YEAR = 2025

df["vehicle_age"] = REFERENCE_YEAR - df["year"]

print(df["vehicle_age"].describe())

df[["year", "vehicle_age"]].head(10)

count    61056.000000
mean        12.310223
std          8.976818
min          0.000000
25%          5.000000
50%         10.000000
75%         18.000000
max         73.000000
Name: vehicle_age, dtype: float64


,year,vehicle_age
0,2022,3
1,2022,3
2,1995,30
3,2012,13
4,1998,27
5,2024,1
6,2022,3
7,2021,4
8,2022,3
9,2006,19


In [3]:
df["mileage_per_year"] = (
    df["mileage_km"] / df["vehicle_age"].clip(lower=1)
)

print(df["mileage_per_year"].describe())

df[
    ["year", "vehicle_age", "mileage_km", "mileage_per_year"]
].head(10)

count     60540.000000
mean      10137.388057
std        9063.464846
min           0.042553
25%        5367.652174
50%        8888.888889
75%       13002.842105
max      800000.000000
Name: mileage_per_year, dtype: float64


,year,vehicle_age,mileage_km,mileage_per_year
0,2022,3,120000.0,40000.000000
1,2022,3,37110.0,12370.000000
2,1995,30,786.0,26.200000
3,2012,13,133000.0,10230.769231
4,1998,27,125225.0,4637.962963
5,2024,1,4100.0,4100.000000
6,2022,3,4917.0,1639.000000
7,2021,4,75000.0,18750.000000
8,2022,3,22000.0,7333.333333
9,2006,19,180000.0,9473.684211


In [4]:
df["is_electric"] = (df["fuel"] == "Electric").astype(int)

print(df["is_electric"].value_counts())

df.loc[
    df["is_electric"] == 1,
    ["fuel", "engine_cc", "battery_kwh", "is_electric"]
].head()

is_electric
0    60585
1      471
Name: count, dtype: int64


,fuel,engine_cc,battery_kwh,is_electric
8,Electric,NaN,71.00,1
236,Electric,NaN,79.00,1
402,Electric,NaN,71.00,1
523,Electric,NaN,55.00,1
542,Electric,NaN,49.92,1


In [5]:
df["mileage_missing"] = df["mileage_km"].isna().astype(int)

print(df["mileage_missing"].value_counts())

mileage_missing
0    60540
1      516
Name: count, dtype: int64


In [6]:
df["engine_cc_missing"] = (
    df["engine_cc"].isna() &
    (df["is_electric"] == 0)
).astype(int)

print(df["engine_cc_missing"].value_counts())

engine_cc_missing
0    60990
1       66
Name: count, dtype: int64


In [7]:
feature_columns = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "brand",
    "model",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

X = df[feature_columns].copy()
y = df["price_pkr"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (61056, 17)
y shape: (61056,)

Features:
['vehicle_age', 'mileage_km', 'mileage_per_year', 'fuel', 'transmission', 'registration_location', 'color', 'assembly', 'body_type', 'engine_cc', 'battery_kwh', 'feature_count', 'brand', 'model', 'is_electric', 'mileage_missing', 'engine_cc_missing']


In [8]:
numerical_features = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

categorical_features = [
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "brand",
    "model"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(numerical_features) + len(categorical_features))

Numerical features: 9
Categorical features: 8
Total: 17


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['vehicle_age', 'mileage_km',
                                  'mileage_per_year', 'engine_cc',
                                  'battery_kwh', 'feature_count', 'is_electric',
                                  'mileage_missing', 'engine_cc_missing']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['fuel', 'transmission',
                                  'registration_location', 'color', 'assembly',
                                  'body_

In [10]:
numerical_features = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

categorical_features = [
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "brand",
    "model"
]

In [11]:
print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Numerical: 9
Categorical: 8
X shape: (61056, 17)
y shape: (61056,)


In [12]:
feature_columns = [
    "vehicle_age",
    "mileage_km",
    "mileage_per_year",
    "fuel",
    "transmission",
    "registration_location",
    "color",
    "assembly",
    "body_type",
    "engine_cc",
    "battery_kwh",
    "feature_count",
    "brand",
    "model",
    "is_electric",
    "mileage_missing",
    "engine_cc_missing"
]

X = df[feature_columns].copy()
y = df["price_pkr"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (61056, 17)
y shape: (61056,)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['vehicle_age', 'mileage_km',
                                  'mileage_per_year', 'engine_cc',
                                  'battery_kwh', 'feature_count', 'is_electric',
                                  'mileage_missing', 'engine_cc_missing']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['fuel', 'transmission',
                                  'registration_location', 'color', 'assembly',
                                  'body_

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (48844, 17)
X_test: (12212, 17)
y_train: (48844,)
y_test: (12212,)


In [15]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_model = DummyRegressor(strategy="median")

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 2491329.102522109
Baseline RMSE: 6171301.401630305
Baseline R²: -0.04850118953024407


In [16]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions))
ridge_r2 = r2_score(y_test, ridge_predictions)

print("Ridge Regression MAE:", ridge_mae)
print("Ridge Regression RMSE:", ridge_rmse)
print("Ridge Regression R²:", ridge_r2)

Ridge Regression MAE: 1983790.1566233146
Ridge Regression RMSE: 4591484.295152414
Ridge Regression R²: 0.4196077311065708


In [18]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

Random Forest MAE: 403280.86175446934
Random Forest RMSE: 1921562.9218816974
Random Forest R²: 0.8983459625162608


In [19]:
from sklearn.ensemble import ExtraTreesRegressor

extra_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

extra_model.fit(X_train, y_train)

extra_predictions = extra_model.predict(X_test)

extra_mae = mean_absolute_error(y_test, extra_predictions)
extra_rmse = np.sqrt(mean_squared_error(y_test, extra_predictions))
extra_r2 = r2_score(y_test, extra_predictions)

print("Extra Trees MAE:", extra_mae)
print("Extra Trees RMSE:", extra_rmse)
print("Extra Trees R²:", extra_r2)

Extra Trees MAE: 375647.2395944172
Extra Trees RMSE: 2032031.2973257068
Extra Trees R²: 0.8863220592742473


In [20]:
y_train_log = np.log1p(y_train)

rf_log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_log_model.fit(X_train, y_train_log)

# Model predicts log prices
rf_log_predictions = rf_log_model.predict(X_test)

# Convert predictions back to PKR
rf_log_predictions_pkr = np.expm1(rf_log_predictions)

rf_log_mae = mean_absolute_error(y_test, rf_log_predictions_pkr)
rf_log_rmse = np.sqrt(
    mean_squared_error(y_test, rf_log_predictions_pkr)
)
rf_log_r2 = r2_score(y_test, rf_log_predictions_pkr)

print("Log Random Forest MAE:", rf_log_mae)
print("Log Random Forest RMSE:", rf_log_rmse)
print("Log Random Forest R²:", rf_log_r2)

Log Random Forest MAE: 394861.37867541553
Log Random Forest RMSE: 2150041.9000084624
Log Random Forest R²: 0.8727349186403408


In [21]:
results = pd.DataFrame({
    "Model": [
        "Median Baseline",
        "Ridge Regression",
        "Random Forest",
        "Extra Trees",
        "Log Random Forest"
    ],
    "MAE": [
        baseline_mae,
        ridge_mae,
        rf_mae,
        extra_mae,
        rf_log_mae
    ],
    "RMSE": [
        baseline_rmse,
        ridge_rmse,
        rf_rmse,
        extra_rmse,
        rf_log_rmse
    ],
    "R2": [
        baseline_r2,
        ridge_r2,
        rf_r2,
        extra_r2,
        rf_log_r2
    ]
})

results = results.sort_values("MAE").reset_index(drop=True)

results

,Model,MAE,RMSE,R2
0,Extra Trees,3.756472e+05,2.032031e+06,0.886322
1,Log Random Forest,3.948614e+05,2.150042e+06,0.872735
2,Random Forest,4.032809e+05,1.921563e+06,0.898346
3,Ridge Regression,1.983790e+06,4.591484e+06,0.419608
4,Median Baseline,2.491329e+06,6.171301e+06,-0.048501


In [22]:
from sklearn.model_selection import cross_val_score

rf_cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

rf_cv_mae = -rf_cv_scores

print("Random Forest CV MAE scores:", rf_cv_mae)
print("Mean CV MAE:", rf_cv_mae.mean())
print("Std CV MAE:", rf_cv_mae.std())

Random Forest CV MAE scores: [534807.10327888 427617.27607228 301400.91970283]
Mean CV MAE: 421275.0996846632
Std CV MAE: 95393.14802047024


In [23]:
extra_cv_scores = cross_val_score(
    extra_model,
    X,
    y,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

extra_cv_mae = -extra_cv_scores

print("Extra Trees CV MAE scores:", extra_cv_mae)
print("Mean CV MAE:", extra_cv_mae.mean())
print("Std CV MAE:", extra_cv_mae.std())

Extra Trees CV MAE scores: [502353.7710087  393774.00029702 305978.65354603]
Mean CV MAE: 400702.1416172534
Std CV MAE: 80319.34613874946


In [24]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    extra_model,
    "../models/used_car_price_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [25]:
print(os.path.exists("../models/used_car_price_model.joblib"))

True


In [26]:
absolute_percentage_errors = (
    np.abs(y_test.values - extra_predictions) / y_test.values
) * 100

print(
    "Median percentage error:",
    np.median(absolute_percentage_errors)
)

print(
    "75th percentile error:",
    np.percentile(absolute_percentage_errors, 75)
)

Median percentage error: 6.17204821264329
75th percentile error: 12.391505457789991


In [27]:
extra_model.fit(X, y)

joblib.dump(
    extra_model,
    "../models/used_car_price_model.joblib"
)

print("Final production model refitted and saved.")

Final production model refitted and saved.
